# TTNet Training for Table Tennis Point Detection

**Upload to Google Drive:**
- `tt_dataset_full/`, `tt_dataset_test/`, `ml/`

**Features:** Keep-alive, local caching (fast!), checkpoint resume

In [ ]:
# Keep-alive (run first!)
from IPython.display import display, HTML
display(HTML('<script>setInterval(()=>{document.querySelector("colab-connect-button").click()},60000)</script>'))

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy data to local SSD (MUCH faster training)
# This takes ~5 min but speeds up each epoch by 3-5x
!echo 'Copying dataset to local...'
!cp -r /content/drive/MyDrive/tt_dataset_full /content/train_data
!cp -r /content/drive/MyDrive/tt_dataset_test /content/test_data
!cp -r /content/drive/MyDrive/ml /content/ml
!echo 'Done!'

In [ ]:
# Configuration (using local paths for speed)
DATASET_PATH = '/content/train_data'
TEST_PATH = '/content/test_data'
ML_PATH = '/content/ml'
OUTPUT_PATH = '/content/drive/MyDrive/tt_models'  # Save to Drive for persistence

BATCH_SIZE = 16
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
RESUME = True

In [ ]:
# Import model
import sys
sys.path.insert(0, ML_PATH)
from ttnet_model import TTNet
import torch, torch.nn as nn
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

In [ ]:
# Dataset
import json, cv2, numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

class TTDataset(Dataset):
    def __init__(self, path, split='train', n_frames=3):
        ann = json.load(open(Path(path)/f'{split}_annotations.json'))
        self.samples = []
        for a in ann:
            fps = a['frame_paths']
            # Update paths to local
            fps = [fp.replace('/Users/conniehuang/Desktop/tt_dataset_full', '/content/train_data')
                     .replace('/Users/conniehuang/Desktop/tt_dataset_test', '/content/test_data') for fp in fps]
            rs, re = a['event_labels']['rally_start'], a['event_labels']['rally_end']
            for i in range(len(fps) - n_frames + 1):
                self.samples.append({'frames': fps[i:i+n_frames], 'label': 1 if rs <= i <= re else 0})
        print(f'{split}: {len(self.samples)} samples')
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        frames = [np.transpose(cv2.cvtColor(cv2.imread(fp), cv2.COLOR_BGR2RGB).astype(np.float32)/255, (2,0,1)) for fp in s['frames']]
        return torch.from_numpy(np.concatenate(frames)), s['label']

train_loader = DataLoader(TTDataset(DATASET_PATH, 'train'), BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(TTDataset(DATASET_PATH, 'val'), BATCH_SIZE, num_workers=4, pin_memory=True)

In [ ]:
# Training with checkpoint resume
import os
from tqdm.notebook import tqdm

device = torch.device('cuda')
model = TTNet(dropout_p=0.5).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

os.makedirs(OUTPUT_PATH, exist_ok=True)
ckpt_path = f'{OUTPUT_PATH}/checkpoint.pth'

start_epoch, best_acc = 0, 0
history = {'loss': [], 'acc': []}

if RESUME and os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optim'])
    start_epoch, best_acc = ckpt['epoch']+1, ckpt['best']
    history = ckpt.get('hist', history)
    print(f'Resuming from epoch {start_epoch}')

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    loss_sum = 0
    for x, y in tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}'):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)['events']
        loss = criterion(torch.log(out/(1-out+1e-8)), y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
    
    model.eval()
    correct = sum((model(x.to(device))['events'].argmax(1)==y.to(device)).sum().item() for x,y in val_loader)
    acc = correct / len(val_loader.dataset)
    scheduler.step(1-acc)
    
    history['loss'].append(loss_sum/len(train_loader))
    history['acc'].append(acc)
    print(f'Epoch {epoch+1}: loss={history["loss"][-1]:.4f}, acc={acc:.4f}')
    
    torch.save({'epoch':epoch,'model':model.state_dict(),'optim':optimizer.state_dict(),'best':max(best_acc,acc),'hist':history}, ckpt_path)
    if acc > best_acc:
        best_acc = acc
        torch.save({'model_state_dict':model.state_dict(),'val_acc':acc}, f'{OUTPUT_PATH}/ttnet_best.pth')
        print('  -> Saved best!')

print(f'Done! Best: {best_acc:.4f}')

In [ ]:
# Plot
import matplotlib.pyplot as plt
fig,(a1,a2)=plt.subplots(1,2,figsize=(10,3))
a1.plot(history['loss']);a1.set_title('Loss')
a2.plot(history['acc']);a2.set_title('Accuracy')
plt.savefig(f'{OUTPUT_PATH}/curves.png');plt.show()

## Test on US Open

In [ ]:
model.load_state_dict(torch.load(f'{OUTPUT_PATH}/ttnet_best.pth')['model_state_dict'])
model.eval()
test_loader = DataLoader(TTDataset(TEST_PATH,'val'), BATCH_SIZE, num_workers=4)
preds,lbls = [],[]
with torch.no_grad():
    for x,y in tqdm(test_loader):
        preds.extend(model(x.to(device))['events'].argmax(1).cpu().numpy())
        lbls.extend(y.numpy())
from sklearn.metrics import classification_report
print(f'Test Accuracy: {sum(p==l for p,l in zip(preds,lbls))/len(lbls):.4f}')
print(classification_report(lbls,preds,target_names=['Not Rally','Rally']))